In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


In [2]:
transform = transforms.Compose([
    transforms.ToTensor()  # converts to [0,1]
])


In [3]:
mnist_train = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)

mnist_test = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

fashion_train = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)

fashion_test = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:00<00:00, 58.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.77MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.36MB/s]
100%|██████████| 26.4M/26.4M [00:01<00:00, 17.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 305kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.60MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.8MB/s]


In [4]:
def torch_to_numpy(dataset):
    X = []
    y = []

    for img, label in dataset:
        X.append(img.view(-1).numpy())  # flatten 28x28 → 784
        y.append(label)

    return np.array(X), np.array(y)


In [5]:
X_train_m, y_train_m = torch_to_numpy(mnist_train)
X_test_m, y_test_m = torch_to_numpy(mnist_test)


In [6]:
X_train_f, y_train_f = torch_to_numpy(fashion_train)
X_test_f, y_test_f = torch_to_numpy(fashion_test)


In [7]:
def standardize(X_train, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test


In [8]:
X_train_m, X_test_m = standardize(X_train_m, X_test_m)
X_train_f, X_test_f = standardize(X_train_f, X_test_f)


In [9]:
def run_svm(X_train, y_train, X_test, y_test,
            kernel='rbf', C=1.0, degree=3, gamma='scale'):

    svm = SVC(kernel=kernel, C=C, degree=degree, gamma=gamma)

    start = time.time()
    svm.fit(X_train, y_train)
    end = time.time()

    y_pred = svm.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    time_ms = (end - start) * 1000

    return acc, time_ms


In [10]:
print("MNIST RESULTS")

acc, t = run_svm(X_train_m, y_train_m, X_test_m, y_test_m,
                 kernel='poly', degree=3)
print(f"Poly Kernel: Accuracy={acc:.4f}, Time={t:.2f} ms")

acc, t = run_svm(X_train_m, y_train_m, X_test_m, y_test_m,
                 kernel='rbf')
print(f"RBF Kernel: Accuracy={acc:.4f}, Time={t:.2f} ms")


MNIST RESULTS
Poly Kernel: Accuracy=0.9611, Time=920192.86 ms
RBF Kernel: Accuracy=0.9661, Time=459909.16 ms


In [11]:
print("\nFashionMNIST RESULTS")

acc, t = run_svm(X_train_f, y_train_f, X_test_f, y_test_f,
                 kernel='poly', degree=3)
print(f"Poly Kernel: Accuracy={acc:.4f}, Time={t:.2f} ms")

acc, t = run_svm(X_train_f, y_train_f, X_test_f, y_test_f,
                 kernel='rbf')
print(f"RBF Kernel: Accuracy={acc:.4f}, Time={t:.2f} ms")



FashionMNIST RESULTS
Poly Kernel: Accuracy=0.8755, Time=552412.08 ms
RBF Kernel: Accuracy=0.8836, Time=448448.43 ms
